# Getting Started with Gendantic

Gendantic is an intelligent synthetic data generation library that combines:
- **Statistical distributions** (numpy-backed) for numeric and categorical fields
- **LLM generation** for semantic content like names, descriptions, and text

This notebook introduces the core concepts and basic usage.

## Setup

Gendantic uses LiteLLM to connect to language models. Configure your environment:

```bash
# Required: LiteLLM proxy URL
export LITELLM_API_BASE="http://localhost:4000"

# Optional: API key for the proxy (if required)
export LITELLM_API_KEY="your-key"

# Optional: Model to use (defaults to gpt-4o-mini)
export LITELLM_MODEL="gpt-4o-mini"
```

Or create a `.env` file in your project root (see `.env.example`).

In [ ]:
from typing import Annotated

import matplotlib.pyplot as plt
from pydantic import BaseModel, Field

from gendantic import (
    generate_synthetic_data,
    Normal,
    Uniform,
    Categorical,
)

plt.style.use('seaborn-v0_8-whitegrid')

## Defining a Model

Gendantic uses standard Pydantic models with `Annotated` types to specify distributions.

- Fields with distribution annotations are **sampled by numpy** (guaranteed statistical properties)
- Fields without distributions are **generated by the LLM** (realistic semantic content)

In [ ]:
class Employee(BaseModel):
    """Employee record with mixed distribution and LLM-generated fields."""
    
    # LLM-generated fields (semantic content)
    first_name: str
    last_name: str
    job_title: str
    bio: str = Field(description="Brief professional bio")
    
    # Distribution-sampled fields (statistical guarantees)
    age: Annotated[int, Uniform(min=22, max=65)]
    salary: Annotated[float, Normal(mean=75000, std=20000)]
    department: Annotated[str, Categorical(weights={
        "Engineering": 0.35,
        "Product": 0.20,
        "Sales": 0.20,
        "Marketing": 0.15,
        "HR": 0.10,
    })]

## Generating Data

Use `generate_synthetic_data()` to create instances of your model:

In [ ]:
# Generate 5 employees
employees = await generate_synthetic_data(Employee, count=5, seed=42)

for emp in employees:
    print(f"{emp.first_name} {emp.last_name}")
    print(f"  {emp.job_title} | {emp.department}")
    print(f"  Age: {emp.age} | Salary: £{emp.salary:,.0f}")
    print(f"  Bio: {emp.bio[:80]}...")
    print()

In [ ]:
# Generate more employees to see the distributions
employees_large = await generate_synthetic_data(Employee, count=100, seed=42)

# Visualize the distribution sampling
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

# Age distribution (Uniform)
ages = [e.age for e in employees_large]
axes[0].hist(ages, bins=15, edgecolor='white', alpha=0.7)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].set_title('Age (Uniform 22-65)')

# Salary distribution (Normal)
salaries = [e.salary for e in employees_large]
axes[1].hist(salaries, bins=15, edgecolor='white', alpha=0.7, color='green')
axes[1].set_xlabel('Salary (£)')
axes[1].set_title('Salary (Normal μ=75k)')

# Department distribution (Categorical)
from collections import Counter
depts = Counter(e.department for e in employees_large)
axes[2].bar(depts.keys(), depts.values(), edgecolor='white', alpha=0.7, color='orange')
axes[2].set_xlabel('Department')
axes[2].set_ylabel('Count')
axes[2].set_title('Department (Categorical)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Reproducibility with Seeds

Setting a `seed` makes distribution sampling deterministic:

In [ ]:
# Generate with the same seed twice
batch1 = await generate_synthetic_data(Employee, count=3, seed=123)
batch2 = await generate_synthetic_data(Employee, count=3, seed=123)

# Distribution-sampled fields will match
for i, (e1, e2) in enumerate(zip(batch1, batch2)):
    print(f"Record {i + 1}:")
    print(f"  Batch 1: age={e1.age}, salary=£{e1.salary:,.0f}, dept={e1.department}")
    print(f"  Batch 2: age={e2.age}, salary=£{e2.salary:,.0f}, dept={e2.department}")
    match = e1.age == e2.age and e1.salary == e2.salary and e1.department == e2.department
    print(f"  Match: {'Yes' if match else 'No'}")
    print()

## Context-Aware Generation

Provide business context for more realistic LLM-generated fields:

In [ ]:
# Silicon Valley startup context
startup_employees = await generate_synthetic_data(
    Employee, 
    count=3, 
    context="Fast-growing Silicon Valley AI startup"
)

print("Silicon Valley Startup:")
for emp in startup_employees:
    print(f"  {emp.first_name} {emp.last_name} - {emp.job_title}")

print()

# Traditional bank context
bank_employees = await generate_synthetic_data(
    Employee, 
    count=3, 
    context="Traditional London investment bank"
)

print("London Investment Bank:")
for emp in bank_employees:
    print(f"  {emp.first_name} {emp.last_name} - {emp.job_title}")

## Next Steps

- **02_distributions.ipynb**: Deep dive into all available distributions
- **03_correlations.ipynb**: Model relationships between fields with copulas
- **04_dynamic_models.ipynb**: Generate models from natural language descriptions
- **05_model_extension.ipynb**: Extend basic models with distributions and correlations